# POP909 Harmonizer — Colab GPU Training
Run all cells top-to-bottom. At the end, `harmonizer_best.pt` will download automatically.

In [ ]:
# 1. Check GPU
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# 2. Install dependencies
!pip install pretty_midi -q

In [ ]:
# 3. Clone POP909 dataset (~200 MB)
import os
if not os.path.exists('POP909-Dataset'):
    !git clone --depth 1 https://github.com/music-x-lab/POP909-Dataset.git
DATA_DIR = 'POP909-Dataset/POP909'
print('Songs found:', len([d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))]))

In [ ]:
# 4. Dataset code (pop909_dataset.py inlined)
import numpy as np
import pretty_midi
import pickle
from torch.utils.data import Dataset, DataLoader

def quantize_to_16th(onset_time, tempo):
    beat_position = onset_time * (tempo / 60.0)
    return int(round(beat_position * 4))

def extract_melody_and_piano(midi_path, tempo=120.0, window_len=64):
    try:
        pm = pretty_midi.PrettyMIDI(midi_path)
        if tempo == 120.0:
            est = pm.estimate_tempo()
            if est > 40:
                tempo = est
        duration_sec = pm.get_end_time()
        duration_16th = int(duration_sec * (tempo / 60.0) * 4) + 1
        melody_seq = np.full(duration_16th, 128, dtype=np.int32)
        piano_seq  = np.full((duration_16th, 4), 0, dtype=np.int32)
        if len(pm.instruments) > 0:
            for note in pm.instruments[0].notes:
                s = min(max(0, quantize_to_16th(note.start, tempo)), duration_16th - 1)
                e = min(max(s + 1, quantize_to_16th(note.end, tempo)), duration_16th)
                melody_seq[s:e] = note.pitch
        if len(pm.instruments) > 2:
            note_times = {}
            for note in pm.instruments[2].notes:
                s = min(max(0, quantize_to_16th(note.start, tempo)), duration_16th - 1)
                e = min(max(s + 1, quantize_to_16th(note.end, tempo)), duration_16th)
                for idx in range(s, e):
                    note_times.setdefault(idx, []).append(note.pitch)
            for idx in range(duration_16th):
                if idx in note_times:
                    pitches = sorted(set(note_times[idx]), reverse=True)[:4]
                    for v, p in enumerate(pitches):
                        piano_seq[idx, v] = p
        return melody_seq, piano_seq
    except Exception as e:
        return None, None

def create_windows(melody_seq, piano_seq, window_len=64, stride=32, rest_threshold=0.8):
    windows = []
    min_len = min(len(melody_seq), len(piano_seq))
    melody_seq, piano_seq = melody_seq[:min_len], piano_seq[:min_len]
    for start in range(0, len(melody_seq) - window_len + 1, stride):
        end = start + window_len
        mel_w = melody_seq[start:end]
        if np.sum(mel_w == 128) / window_len > rest_threshold:
            continue
        windows.append((mel_w.copy(), piano_seq[start:end].copy()))
    return windows

class POP909Dataset(Dataset):
    def __init__(self, data_dir, split='train', val_ratio=0.1, seed=42, cache_file='pop909_cache.pkl'):
        self.split = split
        if os.path.exists(cache_file):
            print(f'Loading cache from {cache_file}')
            with open(cache_file, 'rb') as f:
                self.windows = pickle.load(f)[split]
        else:
            print('Building dataset cache (one-time, ~5 min)...')
            song_folders = sorted([d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))])
            n = len(song_folders)
            np.random.seed(seed)
            val_idx = set(np.random.choice(n, max(1, int(n * val_ratio)), replace=False))
            all_windows = {'train': [], 'val': []}
            for i, folder in enumerate(song_folders):
                midi = os.path.join(data_dir, folder, f'{folder}.mid')
                if not os.path.exists(midi):
                    continue
                mel, pno = extract_melody_and_piano(midi)
                if mel is None:
                    continue
                key = 'val' if i in val_idx else 'train'
                all_windows[key].extend(create_windows(mel, pno))
                if (i + 1) % 100 == 0:
                    print(f'  Processed {i+1}/{n} songs')
            print(f"Train: {len(all_windows['train'])} windows, Val: {len(all_windows['val'])} windows")
            with open(cache_file, 'wb') as f:
                pickle.dump(all_windows, f)
            self.windows = all_windows[split]

    def __len__(self): return len(self.windows)
    def __getitem__(self, idx):
        mel, pno = self.windows[idx]
        return torch.LongTensor(mel), torch.LongTensor(pno)

print('Dataset code ready.')

In [ ]:
# 5. Model code (harmonizer_model.py inlined)
import torch.nn as nn
import torch.nn.functional as F

class MelodyEncoder(nn.Module):
    def __init__(self, embed_dim=128, hidden_dim=256, num_layers=2, dropout=0.3):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.embedding = nn.Embedding(130, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers,
                            batch_first=True, dropout=dropout if num_layers > 1 else 0,
                            bidirectional=True)
    def forward(self, x):
        return self.lstm(self.embedding(x))

class BahdanauAttention(nn.Module):
    def __init__(self, hidden_dim, enc_hidden_dim):
        super().__init__()
        self.q = nn.Linear(hidden_dim, enc_hidden_dim)
        self.k = nn.Linear(enc_hidden_dim, enc_hidden_dim)
        self.v = nn.Linear(enc_hidden_dim, 1)
    def forward(self, dec_h, enc_out):
        scores = self.v(torch.tanh(self.q(dec_h).unsqueeze(1) + self.k(enc_out))).squeeze(-1)
        attn = F.softmax(scores, dim=1)
        ctx = torch.bmm(attn.unsqueeze(1), enc_out).squeeze(1)
        return ctx, attn

class ChordDecoder(nn.Module):
    def __init__(self, embed_dim=128, hidden_dim=256, num_layers=2, dropout=0.3):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.embedding = nn.Embedding(130, embed_dim)
        self.lstm = nn.LSTM(embed_dim * 4, hidden_dim, num_layers=num_layers,
                            batch_first=True, dropout=dropout if num_layers > 1 else 0)
        self.attention = BahdanauAttention(hidden_dim, 2 * hidden_dim)
        self.fc_out = nn.Linear(hidden_dim + 2 * hidden_dim, 4 * 129)

    def _init_hidden(self, enc_h, enc_c, batch_size):
        h = enc_h.view(self.num_layers, 2, batch_size, self.hidden_dim).mean(dim=1)
        c = enc_c.view(self.num_layers, 2, batch_size, self.hidden_dim).mean(dim=1)
        return h, c

    def forward(self, piano_seq, enc_out, enc_hidden, teacher_forcing_ratio=0.5):
        batch_size, seq_len, _ = piano_seq.shape
        dh, dc = self._init_hidden(*enc_hidden, batch_size)
        prev = torch.zeros(batch_size, 4, dtype=torch.long, device=piano_seq.device)
        logits_list, targets_list = [], []
        for t in range(seq_len):
            emb = self.embedding(prev).reshape(batch_size, -1).unsqueeze(1)
            _, (dh, dc) = self.lstm(emb, (dh, dc))
            ctx, _ = self.attention(dh[-1], enc_out)
            logits = self.fc_out(torch.cat([dh[-1], ctx], dim=1))
            logits_list.append(logits)
            targets_list.append(piano_seq[:, t, :])
            if torch.rand(1).item() < teacher_forcing_ratio:
                prev = piano_seq[:, t, :]
            else:
                prev = torch.argmax(logits.view(batch_size, 4, 129), dim=2)
        all_logits = torch.stack(logits_list, dim=1).reshape(batch_size * seq_len, 4 * 129)
        all_targets = torch.stack(targets_list, dim=1).reshape(batch_size * seq_len, 4)
        return all_logits, all_targets

    def generate(self, enc_out, enc_hidden, max_len=64, greedy=True):
        batch_size = enc_out.size(0)
        device = enc_out.device
        dh, dc = self._init_hidden(*enc_hidden, batch_size)
        prev = torch.zeros(batch_size, 4, dtype=torch.long, device=device)
        generated = []
        for _ in range(max_len):
            emb = self.embedding(prev).reshape(batch_size, -1).unsqueeze(1)
            _, (dh, dc) = self.lstm(emb, (dh, dc))
            ctx, _ = self.attention(dh[-1], enc_out)
            logits = self.fc_out(torch.cat([dh[-1], ctx], dim=1)).view(batch_size, 4, 129)
            prev = torch.argmax(logits, dim=2) if greedy else \
                   torch.multinomial(F.softmax(logits, dim=2).reshape(batch_size*4, 129), 1).reshape(batch_size, 4)
            generated.append(prev.detach())
        return torch.stack(generated, dim=1)

class HarmonizerSeq2Seq(nn.Module):
    def __init__(self, embed_dim=128, hidden_dim=256, num_layers=2, dropout=0.3):
        super().__init__()
        self.encoder = MelodyEncoder(embed_dim, hidden_dim, num_layers, dropout)
        self.decoder = ChordDecoder(embed_dim, hidden_dim, num_layers, dropout)
    def forward(self, melody, piano, teacher_forcing_ratio=0.5):
        enc_out, enc_h = self.encoder(melody)
        return self.decoder(piano, enc_out, enc_h, teacher_forcing_ratio)
    def generate(self, melody, max_len=64, greedy=True):
        enc_out, enc_h = self.encoder(melody)
        return self.decoder.generate(enc_out, enc_h, max_len, greedy)

print('Model code ready.')

In [ ]:
# 6. Load datasets
CACHE = 'pop909_cache.pkl'
train_ds = POP909Dataset(DATA_DIR, split='train', cache_file=CACHE)
val_ds   = POP909Dataset(DATA_DIR, split='val',   cache_file=CACHE)
print(f'Train: {len(train_ds)} windows | Val: {len(val_ds)} windows')

In [ ]:
# 7. Train
import json, time
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau

DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EPOCHS     = 15
BATCH_SIZE = 128   # larger batch — GPU can handle it
LR         = 1e-3
CKPT_PATH  = 'harmonizer_best.pt'

torch.manual_seed(42)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

model     = HarmonizerSeq2Seq().to(DEVICE)
optimizer = Adam(model.parameters(), lr=LR)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

CE = nn.CrossEntropyLoss(ignore_index=0)

def run_epoch(loader, train=True, tf_ratio=0.5):
    model.train() if train else model.eval()
    total, n = 0.0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for melody, piano in loader:
            melody, piano = melody.to(DEVICE), piano.to(DEVICE)
            logits, targets = model(melody, piano, teacher_forcing_ratio=tf_ratio)
            lr = logits.view(-1, 4, 129)
            tr = targets.view(-1, 4)
            loss = sum(CE(lr[:, v, :], tr[:, v]) for v in range(4))
            if train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            total += loss.item(); n += 1
    return total / n

losses = {'train': [], 'val': []}
best_val, best_epoch = float('inf'), 0

print(f'Training on {DEVICE} for {EPOCHS} epochs...')
for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    tr_loss = run_epoch(train_loader, train=True,  tf_ratio=0.5)
    va_loss = run_epoch(val_loader,   train=False, tf_ratio=1.0)
    scheduler.step(va_loss)
    losses['train'].append(tr_loss)
    losses['val'].append(va_loss)
    marker = '  <-- best' if va_loss < best_val else ''
    print(f'Epoch {epoch:2d}/{EPOCHS}  train={tr_loss:.4f}  val={va_loss:.4f}  ({time.time()-t0:.0f}s){marker}')
    if va_loss < best_val:
        best_val, best_epoch = va_loss, epoch
        torch.save({
            'epoch': epoch, 'val_loss': va_loss,
            'model_state': model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'hyperparams': {'embed_dim':128,'hidden_dim':256,'num_layers':2,'dropout':0.3}
        }, CKPT_PATH)

with open('harmonizer_losses.json', 'w') as f:
    json.dump(losses, f, indent=2)
print(f'\nDone. Best val loss {best_val:.4f} at epoch {best_epoch}.')

In [ ]:
# 8. Download checkpoint + losses
from google.colab import files
files.download('harmonizer_best.pt')
files.download('harmonizer_losses.json')